# 23-18 · Группируем файлы в дубликаты

Практика к разделу [«Находим группы дубликатов»](../../site/chapters/glava-23/23-19-gruppy-dublikatov.html). Настоящий файл — `projects/python/safesort/src/safesort/duplicates.py`.

## Цель

Воспроизвести двухэтапную логику `find_duplicates()` (сначала группировка по размеру, потом по дайджесту) на синтетических записях, чтобы не создавать настоящие файлы на диске.

## Example

In [ ]:
import hashlib
from collections import defaultdict
from dataclasses import dataclass


@dataclass(frozen=True)
class FileInfo:
    name: str
    size: int
    content: bytes


def find_duplicates(files):
    by_size = defaultdict(list)
    for file in files:
        by_size[file.size].append(file)

    groups = []
    for size, candidates in by_size.items():
        if len(candidates) < 2:
            continue
        by_digest = defaultdict(list)
        for candidate in candidates:
            digest = hashlib.sha256(candidate.content).hexdigest()
            by_digest[digest].append(candidate)
        for digest, matched in by_digest.items():
            if len(matched) >= 2:
                groups.append({"size": size, "digest": digest, "files": tuple(matched)})
    return groups


fajly = [
    FileInfo("notes.txt", 8, b"AAAAAAAA"),
    FileInfo("copy_of_notes.txt", 8, b"AAAAAAAA"),
    FileInfo("unikalnyj.txt", 8, b"BBBBBBBB"),   # тот же размер, другое содержимое
    FileInfo("photo1.jpg", 100, b"J" * 100),
    FileInfo("photo2.jpg", 100, b"J" * 100),
    FileInfo("odinokij.pdf", 55, b"P" * 55),      # уникальный размер — не может быть дубликатом
]

gruppy = find_duplicates(fajly)
for g in gruppy:
    print(g["size"], g["digest"][:12], [f.name for f in g["files"]])

## Проверка результата

In [ ]:
imena_v_gruppah = {frozenset(f.name for f in g["files"]) for g in gruppy}

assert len(gruppy) == 2
assert frozenset({"notes.txt", "copy_of_notes.txt"}) in imena_v_gruppah
assert frozenset({"photo1.jpg", "photo2.jpg"}) in imena_v_gruppah
assert not any("odinokij.pdf" in imena for imena in imena_v_gruppah)
print("Верно: найдены ровно две группы дубликатов, уникальные файлы не попали ни в одну.")

## Starter

Заполните отмеченное место. Неизменённый starter не проходит tests.

In [ ]:
def gruppy_s_pustoj_paroj(files):
    # TODO: append two zero-byte FileInfo values, then find duplicates.
    raise NotImplementedError


gruppy2 = gruppy_s_pustoj_paroj(fajly)

## Task

Добавьте к списку два пустых файла и верните результат `find_duplicates()`.

## Tests

Запустите после task cell: есть основной пример и хотя бы один крайний случай.

In [ ]:
gruppa_pustyh = next(g for g in gruppy2 if g["size"] == 0)
assert {f.name for f in gruppa_pustyh["files"]} == {"pustoj_a.txt", "pustoj_b.txt"}
assert gruppa_pustyh["digest"] == hashlib.sha256(b"").hexdigest()
assert not any(g["size"] == 0 for g in find_duplicates(fajly + [FileInfo("one", 0, b"")]))
print("Tests passed")

## Hint

Создайте два `FileInfo` с size 0 и content `b""`.

## Solution

<details><summary>Показать решение после собственной попытки</summary>

```python
def gruppy_s_pustoj_paroj(files):
    return find_duplicates(files + [
        FileInfo("pustoj_a.txt", 0, b""),
        FileInfo("pustoj_b.txt", 0, b""),
    ])


gruppy2 = gruppy_s_pustoj_paroj(fajly)
```

</details>